# XGBoost Model — Exploratory Data Analysis
## Predicting Player Performance Decline Across Multiple Competitions
### Seasons: 2022-2023 · 2023-2024 · 2024-2025

**Goal**: Identify and combine all available data (fixtures, rest days, congestion, minutes played, per-match stats, injury impact) to build a dataset for predicting performance decline in players competing in multiple competitions — with focus on Champions League teams.


## Section 1 — Setup & Directory Inventory
Scan the full `/data` directory and catalogue every CSV file available.


In [71]:
import os, glob
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', 200)

DATA_DIR = '/tmp/fixture-iq-repo/XgBoost_model/data'
SEASONS   = ['2022-2023', '2023-2024', '2024-2025']

print("=" * 90)
print("FULL DIRECTORY INVENTORY")
print("=" * 90)

for season in SEASONS:
    season_path = os.path.join(DATA_DIR, season)
    print(f"\n📅 {season}/")
    for subdir in sorted(os.listdir(season_path)):
        subpath = os.path.join(season_path, subdir)
        if os.path.isdir(subpath):
            # only show top-level CSVs + sub-CSVs (not per-match reports)
            csvs = glob.glob(os.path.join(subpath, '*.csv'))
            sub_csvs = glob.glob(os.path.join(subpath, '*', '*.csv'))
            top_level_subs = sorted({os.path.basename(os.path.dirname(f)) for f in sub_csvs})
            print(f"  ├── {subdir}/")
            for c in sorted(csvs):
                sz = os.path.getsize(c) / 1024
                print(f"  │     {os.path.basename(c):55s}  {sz:7.0f} KB")
            for s in top_level_subs:
                sub_files = glob.glob(os.path.join(subpath, s, '*.csv'))
                print(f"  │     {s}/ ({len(sub_files)} csvs)")

print(f"\n📁 API Seasons/")
for d in sorted(os.listdir(DATA_DIR)):
    if d.startswith('API_'):
        api_files = sorted(glob.glob(os.path.join(DATA_DIR, d, '*.csv')))
        print(f"  ├── {d}/")
        for f in api_files:
            sz = os.path.getsize(f) / 1024
            print(f"  │     {os.path.basename(f):60s}  {sz:7.0f} KB")


FULL DIRECTORY INVENTORY

📅 2022-2023/
  ├── fbref/
  │     chelsea_2022_2023/ (3 csvs)
  │     liverpool_2022_2023/ (3 csvs)
  │     manchester_city_2022_2023/ (3 csvs)
  │     tottenham_hotspur_2022_2023/ (3 csvs)
  ├── injuries/
  │     ALL_TEAMS_2022-2023_injuries_days_out.csv                    221 KB
  │     estimated_injury_spells_2022-2023.csv                         34 KB
  │     player_season_absence_burden_2022-2023.csv                    33 KB
  │     team_match_injury_outcomes_2022-2023.csv                      43 KB
  │     team_season_injury_burden_2022-2023.csv                        1 KB
  ├── sofascore/
  │     champions_league/ (1 csvs)
  │     premier_league/ (1 csvs)
  ├── sofascore_dynamic/
  │     fixtureiq_dynamic_analytics_clean.csv                        730 KB
  │     fixtureiq_dynamic_master.csv                                5971 KB
  ├── sofascore_raw_pl_centric/

📅 2023-2024/
  ├── fbref/
  │     arsenal_2023_2024/ (3 csvs)
  │     manchester_city_2023_20

## Section 2 — Data Source A: API Multi-Competition Player Stats
Per-match player stats across **all competitions** (Premier League, Champions League, FA Cup, League Cup).
This is the **widest coverage** dataset — all 20 PL teams, all competitions, 3 seasons.


In [72]:
# ── A1: API Multi-Competition Player Stats ──────────────────────────────────
# Best source: per-match stats for every player across ALL competitions, all 20 PL teams

api_player_dfs = {}
for season in SEASONS:
    season_key = season.replace('-', '_')
    path = os.path.join(DATA_DIR, f'API_SEASON_{season_key}',
                        f'multi_competition_player_stats_{season_key}.csv')
    if os.path.exists(path):
        df = pd.read_csv(path, parse_dates=['date'])
        api_player_dfs[season] = df
        print(f"✓ {season}: {df.shape[0]:,} rows × {df.shape[1]} cols  |  {os.path.getsize(path)/1e6:.1f} MB")

# Show full column info from the first available season
sample = next(iter(api_player_dfs.values()))
print(f"\n{'─'*80}")
print(f"COLUMNS ({sample.shape[1]} total)  ←  same across all 3 seasons")
print(f"{'─'*80}")
for i, col in enumerate(sample.columns, 1):
    print(f"  {i:2d}. {col:<35}  dtype: {sample[col].dtype}")

print(f"\n{'─'*80}")
print("SAMPLE ROWS (2 rows)")
print(f"{'─'*80}")
print(sample.head(2).to_string())

print(f"\n{'─'*80}")
print("COMPETITIONS COVERED")
print(f"{'─'*80}")
for season, df in api_player_dfs.items():
    comps = df['competition'].value_counts()
    print(f"\n  {season}:")
    for comp, cnt in comps.items():
        matches = df[df['competition'] == comp]['fixture_id'].nunique()
        print(f"    {comp:<30}  {cnt:5,} player-rows  ({matches} matches)")


✓ 2022-2023: 23,455 rows × 34 cols  |  4.3 MB
✓ 2023-2024: 21,942 rows × 34 cols  |  4.0 MB
✓ 2024-2025: 23,246 rows × 34 cols  |  4.3 MB

────────────────────────────────────────────────────────────────────────────────
COLUMNS (34 total)  ←  same across all 3 seasons
────────────────────────────────────────────────────────────────────────────────
   1. fixture_id                           dtype: int64
   2. date                                 dtype: datetime64[ns, UTC]
   3. competition                          dtype: object
   4. season                               dtype: int64
   5. round                                dtype: object
   6. home_team                            dtype: object
   7. away_team                            dtype: object
   8. player_team                          dtype: object
   9. player_id                            dtype: int64
  10. player_name                          dtype: object
  11. player_number                        dtype: int64
  12. player_p

## Section 3 — Data Source B: API Team Results (with Match Stats)
Per-match **team-level** stats including possession, shots, xG for both home and away.
Used to add match context (was it a tough opponent? high-intensity game?) to each player row.


In [73]:
# ── B1: API Multi-Competition Team Results ───────────────────────────────────
# Per-match team stats for both sides: xG, possession, shots, fouls, cards.
# merge key: fixture_id  (same fixture_id as api_player_dfs)

api_team_dfs = {}
for season in SEASONS:
    season_key = season.replace('-', '_')
    path = os.path.join(DATA_DIR, f'API_SEASON_{season_key}',
                        f'multi_competition_team_results_{season_key}.csv')
    if os.path.exists(path):
        df = pd.read_csv(path, parse_dates=['date'])
        api_team_dfs[season] = df
        print(f"✓ {season}: {df.shape[0]:,} rows × {df.shape[1]} cols  |  {os.path.getsize(path)/1024:.0f} KB")

sample = next(iter(api_team_dfs.values()))
print(f"\n{'─'*80}")
print(f"COLUMNS ({sample.shape[1]} total)")
print(f"{'─'*80}")
for i, col in enumerate(sample.columns, 1):
    print(f"  {i:2d}. {col:<45}  dtype: {sample[col].dtype}")

print(f"\n{'─'*80}")
print("SAMPLE ROWS (2 rows)")
print(f"{'─'*80}")
print(sample.head(2).to_string())

print(f"\n{'─'*80}")
print("COMPETITIONS COVERED")
print(f"{'─'*80}")
for season, df in api_team_dfs.items():
    comps = df['competition'].value_counts()
    print(f"\n  {season}:")
    for comp, cnt in comps.items():
        matches = df[df['competition'] == comp]['fixture_id'].nunique()
        print(f"    {comp:<30}  {cnt:4,} team-rows  ({matches} matches)")

print(f"\n{'─'*80}")
print("NULL CHECK — key columns")
print(f"{'─'*80}")
for col in ['home_expected_goals', 'away_expected_goals', 'home_ball_possession',
            'home_total_passes', 'home_total_shots']:
    nulls = sample[col].isna().sum()
    pct = nulls / len(sample) * 100
    print(f"  {col:<35}  nulls: {nulls:4}  ({pct:.1f}%)")


✓ 2022-2023: 2,802 rows × 49 cols  |  485 KB
✓ 2023-2024: 2,722 rows × 51 cols  |  477 KB
✓ 2024-2025: 2,784 rows × 51 cols  |  495 KB

────────────────────────────────────────────────────────────────────────────────
COLUMNS (49 total)
────────────────────────────────────────────────────────────────────────────────
   1. fixture_id                                     dtype: int64
   2. date                                           dtype: datetime64[ns, UTC]
   3. competition                                    dtype: object
   4. season                                         dtype: int64
   5. round                                          dtype: object
   6. team_name                                      dtype: object
   7. team_id                                        dtype: int64
   8. is_home                                        dtype: bool
   9. opponent                                       dtype: object
  10. opponent_id                                    dtype: int64
  11. 

## Section 4 — Data Source C: SofaScore Dynamic Analytics
Per-match **workload & fatigue metrics** calculated dynamically for every player.
Covers all 20 PL teams. **Only available for Premier League fixtures.**
Key fields: `rest_days`, `high_congestion_flag`, `min_last_7d`, `acwr_ratio`, `consecutive_away_games`.


In [74]:
# ── C1: SofaScore Dynamic Analytics (clean version — 19 cols) ────────────────
# Per-match workload/fatigue. Premier League only, all 20 teams.

sofascore_dynamic = {}
for season in SEASONS:
    path = os.path.join(DATA_DIR, season, 'sofascore_dynamic',
                        'fixtureiq_dynamic_analytics_clean.csv')
    if os.path.exists(path):
        df = pd.read_csv(path, parse_dates=['match_date_str'])
        sofascore_dynamic[season] = df
        print(f"✓ {season}: {df.shape[0]:,} rows × {df.shape[1]} cols")

sample = next(iter(sofascore_dynamic.values()))
print(f"\n{'─'*80}")
print(f"COLUMNS ({sample.shape[1]} total)")
print(f"{'─'*80}")
for i, col in enumerate(sample.columns, 1):
    print(f"  {i:2d}. {col:<35}  dtype: {sample[col].dtype}")

print(f"\n{'─'*80}")
print("SAMPLE ROWS (3 rows)")
print(f"{'─'*80}")
print(sample.head(3).to_string())

print(f"\n{'─'*80}")
print("KEY METRIC RANGES (2022-2023)")
print(f"{'─'*80}")
for col in ['rest_days', 'min_last_7d', 'acwr_ratio', 'consecutive_away_games',
            'minutesPlayed', 'rating', 'high_congestion_flag']:
    s = sample[col]
    print(f"  {col:<30}  min={s.min():.1f}  max={s.max():.1f}  mean={s.mean():.2f}  nulls={s.isna().sum()}")


✓ 2022-2023: 6,500 rows × 19 cols
✓ 2023-2024: 13,142 rows × 17 cols
✓ 2024-2025: 6,726 rows × 19 cols

────────────────────────────────────────────────────────────────────────────────
COLUMNS (19 total)
────────────────────────────────────────────────────────────────────────────────
   1. match_date_str                       dtype: datetime64[ns]
   2. match_id                             dtype: int64
   3. competition                          dtype: object
   4. teamName                             dtype: object
   5. name                                 dtype: object
   6. position                             dtype: object
   7. rating                               dtype: float64
   8. elo                                  dtype: float64
   9. is_away                              dtype: int64
  10. consecutive_away_games               dtype: int64
  11. minutesPlayed                        dtype: int64
  12. rest_days                            dtype: int64
  13. high_congestion_flag

In [75]:
# ── C2: SofaScore Dynamic Master (108 cols — full per-match detail) ──────────
# Same rows as clean version but with ALL raw SofaScore per-match stats too.

sofascore_master = {}
for season in SEASONS:
    path = os.path.join(DATA_DIR, season, 'sofascore_dynamic',
                        'fixtureiq_dynamic_master.csv')
    if os.path.exists(path):
        df = pd.read_csv(path)
        sofascore_master[season] = df
        print(f"✓ {season}: {df.shape[0]:,} rows × {df.shape[1]} cols")

sample_master = next(iter(sofascore_master.values()))
print(f"\n{'─'*80}")
print(f"ALL COLUMNS ({sample_master.shape[1]} total)")
print(f"{'─'*80}")
for i, col in enumerate(sample_master.columns, 1):
    print(f"  {i:3d}. {col:<45}  dtype: {sample_master[col].dtype}")


✓ 2022-2023: 7,304 rows × 108 cols
✓ 2023-2024: 14,758 rows × 106 cols
✓ 2024-2025: 7,587 rows × 117 cols

────────────────────────────────────────────────────────────────────────────────
ALL COLUMNS (108 total)
────────────────────────────────────────────────────────────────────────────────
    1. date                                           dtype: object
    2. name                                           dtype: object
    3. slug                                           dtype: object
    4. shortName                                      dtype: object
    5. position                                       dtype: object
    6. jerseyNumber                                   dtype: float64
    7. height                                         dtype: int64
    8. userCount                                      dtype: int64
    9. gender                                         dtype: object
   10. country                                        dtype: object
   11. id                   

## Section 5 — Data Source D: FBRef Detailed Match Stats
Per-match **action stats** (goals, shots, tackles, crosses, fouls, cards…) scraped from FBRef.
**Coverage is limited to 4 CL teams per season** (teams were selected because they played Champions League).


In [76]:
# ── D1: FBRef Master Player Stats per match (39 cols) ────────────────────────
# Per-match action stats. One file per team, per season.
# merge key: (Player, Team, match_date)

fbref_teams = {
    '2022-2023': ['chelsea_2022_2023', 'liverpool_2022_2023',
                  'manchester_city_2022_2023', 'tottenham_hotspur_2022_2023'],
    '2023-2024': ['arsenal_2023_2024', 'manchester_city_2023_2024',
                  'manchester_united_2023_2024', 'newcastle_united_2023_2024'],
    '2024-2025': ['arsenal_2024_2025', 'aston_villa_2024_2025',
                  'liverpool_2024_2025', 'manchester_city_2024_2025'],
}

fbref_dfs = {}
for season, teams in fbref_teams.items():
    season_rows = []
    for team in teams:
        path = os.path.join(DATA_DIR, season, 'fbref', team,
                            'match_reports', 'master_player_stats.csv')
        if os.path.exists(path):
            df = pd.read_csv(path, parse_dates=['match_date'])
            season_rows.append(df)
    if season_rows:
        combined = pd.concat(season_rows, ignore_index=True)
        fbref_dfs[season] = combined
        print(f"✓ {season}: {combined.shape[0]:,} rows × {combined.shape[1]} cols  |  teams: {', '.join(teams)}")

sample_fbref = next(iter(fbref_dfs.values()))
print(f"\n{'─'*80}")
print(f"COLUMNS ({sample_fbref.shape[1]} total)")
print(f"{'─'*80}")
for i, col in enumerate(sample_fbref.columns, 1):
    print(f"  {i:2d}. {col:<35}  dtype: {sample_fbref[col].dtype}")

print(f"\n{'─'*80}")
print("SAMPLE ROWS (2 rows)")
print(f"{'─'*80}")
print(sample_fbref.head(2).to_string())

print(f"\n{'─'*80}")
print("COMPETITIONS COVERED IN FBREF DATA")
print(f"{'─'*80}")
for season, df in fbref_dfs.items():
    comps = df['competition'].value_counts()
    print(f"\n  {season}:")
    for comp, cnt in comps.items():
        print(f"    {comp:<30}  {cnt:4} player-rows")


✓ 2022-2023: 3,187 rows × 39 cols  |  teams: chelsea_2022_2023, liverpool_2022_2023, manchester_city_2022_2023, tottenham_hotspur_2022_2023
✓ 2023-2024: 3,116 rows × 39 cols  |  teams: arsenal_2023_2024, manchester_city_2023_2024, manchester_united_2023_2024, newcastle_united_2023_2024
✓ 2024-2025: 3,436 rows × 39 cols  |  teams: arsenal_2024_2025, aston_villa_2024_2025, liverpool_2024_2025, manchester_city_2024_2025

────────────────────────────────────────────────────────────────────────────────
COLUMNS (39 total)
────────────────────────────────────────────────────────────────────────────────
   1. season                               dtype: object
   2. tracked_team                         dtype: object
   3. match_date                           dtype: datetime64[ns]
   4. competition                          dtype: object
   5. opponent_team                        dtype: object
   6. original_file                        dtype: object
   7. Match_ID                             dtyp

In [77]:
# ── D2: FBRef Season Aggregates per player (all competitions) ─────────────────
# Season-level totals per player. Not per-match.
# merge key: (Player, Team, season)

sample_team_dir = os.path.join(DATA_DIR, '2022-2023', 'fbref', 'chelsea_2022_2023')
path_all_comps = os.path.join(sample_team_dir, 'chelsea_2022_2023_players_all_competitions.csv')
path_matches   = os.path.join(sample_team_dir, 'chelsea_2022_2023_matches_all.csv')

df_all_comps = pd.read_csv(path_all_comps)
df_matches   = pd.read_csv(path_matches, parse_dates=['Date'])

print("── players_all_competitions.csv (season aggregate per player) ──")
print(f"Shape: {df_all_comps.shape[0]} rows × {df_all_comps.shape[1]} cols")
print(f"Columns: {list(df_all_comps.columns)}")
print(df_all_comps.head(3).to_string())

print(f"\n{'─'*80}")
print("── matches_all.csv (full fixture list for the team) ──")
print(f"Shape: {df_matches.shape[0]} rows × {df_matches.shape[1]} cols")
print(f"Columns: {list(df_matches.columns)}")
print(df_matches.head(3).to_string())


── players_all_competitions.csv (season aggregate per player) ──
Shape: 41 rows × 22 cols
Columns: ['Player', 'Nation', 'Pos', 'Age', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'Gls_90', 'Ast_90', 'G+A_90', 'G-PK_90', 'G+A-PK_90', 'Matches']
              Player Nation    Pos  Age  MP  Starts    Min   90s  Gls  Ast   G+A  G-PK   PK  PKatt  CrdY  CrdR  Gls_90  Ast_90  G+A_90  G-PK_90  G+A-PK_90  Matches
0  Kepa Arrizabalaga  esESP     GK   27  39      39  3,465  38.5  0.0  0.0   0.0   0.0  0.0    0.0   3.0   0.0    0.00    0.00    0.00     0.00       0.00  Matches
1        Kai Havertz  deGER  FW,MF   23  47      38  3,249  36.1  9.0  1.0  10.0   7.0  2.0    3.0   5.0   0.0    0.25    0.03    0.28     0.19       0.22  Matches
2       Thiago Silva  brBRA     DF   37  35      33  2,992  33.2  0.0  2.0   2.0   0.0  0.0    0.0   4.0   0.0    0.00    0.06    0.06     0.00       0.06  Matches

──────────────────────────────────────────────────────

## Section 6 — Data Source E: Injury Data (5 files)
Five complementary injury files at different granularities:
- **ALL_TEAMS injuries_days_out**: Row per player per missed fixture (most granular)
- **estimated_injury_spells**: Row per injury episode with type & dates
- **player_season_absence_burden**: Season totals per player
- **team_match_injury_outcomes**: Per-match team injury burden (squad missing players count)
- **team_season_injury_burden**: Season totals per team


In [78]:
# ── E: All 5 Injury Files ─────────────────────────────────────────────────────

injury_files = {
    'ALL_TEAMS_injuries_days_out':   'ALL_TEAMS_{season}_injuries_days_out.csv',
    'estimated_injury_spells':       'estimated_injury_spells_{season}.csv',
    'player_season_absence_burden':  'player_season_absence_burden_{season}.csv',
    'team_match_injury_outcomes':    'team_match_injury_outcomes_{season}.csv',
    'team_season_injury_burden':     'team_season_injury_burden_{season}.csv',
}

injury_data = {}  # {file_key: {season: df}}

for file_key, template in injury_files.items():
    injury_data[file_key] = {}
    for season in SEASONS:
        filename = template.format(season=season)
        path = os.path.join(DATA_DIR, season, 'injuries', filename)
        if os.path.exists(path):
            df = pd.read_csv(path)
            injury_data[file_key][season] = df

# Show columns and samples for each file
for file_key, seasons_dict in injury_data.items():
    if not seasons_dict:
        continue
    sample = next(iter(seasons_dict.values()))
    seasons_available = list(seasons_dict.keys())
    total_rows = sum(df.shape[0] for df in seasons_dict.values())

    print(f"\n{'═'*80}")
    print(f"📋 {file_key}")
    print(f"   Seasons available: {seasons_available}  |  Total rows (all seasons): {total_rows:,}")
    print(f"   Shape per file: {sample.shape[0]} rows × {sample.shape[1]} cols")
    print(f"{'─'*80}")
    print(f"   COLUMNS:")
    for i, col in enumerate(sample.columns, 1):
        print(f"     {i:2d}. {col:<40}  dtype: {sample[col].dtype}")
    print(f"\n   SAMPLE (2 rows):")
    print(sample.head(2).to_string())



════════════════════════════════════════════════════════════════════════════════
📋 ALL_TEAMS_injuries_days_out
   Seasons available: ['2022-2023', '2023-2024', '2024-2025']  |  Total rows (all seasons): 9,526
   Shape per file: 2773 rows × 9 cols
────────────────────────────────────────────────────────────────────────────────
   COLUMNS:
      1. season                                    dtype: object
      2. team_name                                 dtype: object
      3. player_id                                 dtype: int64
      4. player_name                               dtype: object
      5. injury_type                               dtype: object
      6. reason                                    dtype: object
      7. fixture_date                              dtype: object
      8. end_date                                  dtype: float64
      9. days_out                                  dtype: int64

   SAMPLE (2 rows):
      season team_name  player_id player_name      inj

## Section 7 — Complete Data Inventory Summary
A consolidated table of every dataset: rows, columns, what it provides, and the merge keys available.


In [79]:
# ── Full Inventory Summary Table ──────────────────────────────────────────────

rows = []

# A — API Player Stats
for season, df in api_player_dfs.items():
    rows.append({'Season': season, 'Source': 'API Player Stats',
                 'Granularity': 'per-match, per-player',
                 'Coverage': 'All 20 PL teams + CL/FA Cup/LC',
                 'Rows': df.shape[0], 'Cols': df.shape[1],
                 'Merge Keys': 'fixture_id, date, player_name, player_team'})

# B — API Team Results
for season, df in api_team_dfs.items():
    rows.append({'Season': season, 'Source': 'API Team Results',
                 'Granularity': 'per-match, per-team',
                 'Coverage': 'All 20 PL teams + CL/FA Cup/LC',
                 'Rows': df.shape[0], 'Cols': df.shape[1],
                 'Merge Keys': 'fixture_id, date, team_name'})

# C — SofaScore Dynamic Clean
for season, df in sofascore_dynamic.items():
    rows.append({'Season': season, 'Source': 'SofaScore Dynamic (clean)',
                 'Granularity': 'per-match, per-player',
                 'Coverage': 'All 20 PL teams — PL only',
                 'Rows': df.shape[0], 'Cols': df.shape[1],
                 'Merge Keys': 'match_date_str, teamName, name'})

# C2 — SofaScore Master
for season, df in sofascore_master.items():
    rows.append({'Season': season, 'Source': 'SofaScore Master (full)',
                 'Granularity': 'per-match, per-player',
                 'Coverage': 'All 20 PL teams — PL only',
                 'Rows': df.shape[0], 'Cols': df.shape[1],
                 'Merge Keys': 'match_date_str, teamName, name'})

# D — FBRef Match Stats
for season, df in fbref_dfs.items():
    rows.append({'Season': season, 'Source': 'FBRef Match Stats',
                 'Granularity': 'per-match, per-player',
                 'Coverage': f'4 CL teams  ({", ".join(fbref_teams[season][:2])}…)',
                 'Rows': df.shape[0], 'Cols': df.shape[1],
                 'Merge Keys': 'match_date, Player, Team'})

# E — Injury files
injury_labels = {
    'ALL_TEAMS_injuries_days_out':  ('per-fixture absence', 'All 20 teams', 'team_name, player_name, fixture_date'),
    'estimated_injury_spells':      ('per injury spell',    'All 20 teams', 'Team, Player, First_Missed_Fixture'),
    'player_season_absence_burden': ('season totals',       'All 20 teams', 'Team, Player, Season'),
    'team_match_injury_outcomes':   ('per-match team burden','All 20 teams', 'Team, Fixture_Date'),
    'team_season_injury_burden':    ('season totals team',  'All 20 teams', 'Team, Season'),
}
for file_key, seasons_dict in injury_data.items():
    gran, cov, keys = injury_labels[file_key]
    for season, df in seasons_dict.items():
        rows.append({'Season': season, 'Source': f'Injury: {file_key}',
                     'Granularity': gran, 'Coverage': cov,
                     'Rows': df.shape[0], 'Cols': df.shape[1],
                     'Merge Keys': keys})

summary_df = pd.DataFrame(rows)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 55)
print(summary_df.to_string(index=False))


   Season                               Source           Granularity                                                    Coverage  Rows  Cols                                 Merge Keys
2022-2023                     API Player Stats per-match, per-player                              All 20 PL teams + CL/FA Cup/LC 23455    34 fixture_id, date, player_name, player_team
2023-2024                     API Player Stats per-match, per-player                              All 20 PL teams + CL/FA Cup/LC 21942    34 fixture_id, date, player_name, player_team
2024-2025                     API Player Stats per-match, per-player                              All 20 PL teams + CL/FA Cup/LC 23246    34 fixture_id, date, player_name, player_team
2022-2023                     API Team Results   per-match, per-team                              All 20 PL teams + CL/FA Cup/LC  2802    49                fixture_id, date, team_name
2023-2024                     API Team Results   per-match, per-team            

## Section 8 — Integration Plan: Building the Master Training Table

**Objective**: One row = one player × one match, enriched with all available context.

### Primary Merge: API Player Stats → base table
The `multi_competition_player_stats` file is the base because it covers:
- All 3 seasons
- All 20 PL teams
- All competitions (PL + CL + FA/LC)

### Enrichments via LEFT JOIN:

| JOIN | Key | What it adds |
|------|-----|-------------|
| **API Team Results** | `fixture_id` | Match-level team stats (xG, possession, shots) |
| **SofaScore Dynamic** | `(date, player_name, team)` | `rest_days`, `acwr_ratio`, `min_last_7d`, `high_congestion_flag` |
| **FBRef Match Stats** | `(match_date, player, team)` | `shots`, `tackles_won`, `crosses`, `fouls`, `interceptions` |
| **Injury: team_match_outcomes** | `(date, team)` | Squad injury burden on that matchday |
| **Injury: ALL_TEAMS_days_out** | `(fixture_date, player_name, team)` | Was *this specific player* injured around this date? |

### Target Variable Options:
- **Regression**: `rating` of next match (performance level)
- **Classification**: Binary flag — rating drops ≥ 1 point in next match
- **Multi-horizon**: Predict decline over next 3 matches (rolling window)

### Expected Master Table:
~70,000–90,000 rows × ~60–70 columns across all 3 seasons


## Section 9 — Build Master Training Table → `Fixture_IQ_Data_Seasons.csv`

Executes all 7 build steps end-to-end:

| Step | Action | Join Key |
|------|--------|----------|
| 1 | Stack API Player Stats (base, 3 seasons) | — |
| 2 | LEFT JOIN API Team Results | `fixture_id + team` |
| 3 | LEFT JOIN SofaScore Fatigue Metrics | `date + player + team` |
| 4 | LEFT JOIN FBRef Action Stats | `date + player + team` |
| 5 | LEFT JOIN Squad Injury Burden | `date + team` |
| 6 | LEFT JOIN Individual Injury Flag | `date + player + team` |
| 7 | Engineer target variable | `next_sofascore_rating` per player |


In [25]:
# ============================================================
# SECTION 9 — Build Master Training Table
# Output: Fixture_IQ_Data_Seasons_2022-2025.csv
# One row = one player × one match, all 3 seasons, all comps
# ============================================================

import unicodedata, os
import pandas as pd
import numpy as np

DATA_DIR = '/tmp/fixture-iq-repo/XgBoost_model/data'
SEASONS  = ['2022-2023', '2023-2024', '2024-2025']
OUT_PATH = '/tmp/fixture-iq-repo/XgBoost_model/Fixture_IQ_Data_Seasons_2022-2025.csv'

# ── utilities ────────────────────────────────────────────────────────────────
def norm(s):
    """Lowercase + strip accents + remove punctuation → consistent join key."""
    if pd.isna(s): return ''
    s = unicodedata.normalize('NFD', str(s).lower().strip())
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.replace("'", '').replace('-', ' ').replace('.', '').strip()

def to_date(series):
    """Parse any date string/datetime to tz-naive midnight-normalised date."""
    dt = pd.to_datetime(series, errors='coerce', utc=True)
    return dt.dt.tz_convert(None).dt.normalize()

def pct_float(series):
    """Convert '56%' strings → 56.0 floats."""
    return pd.to_numeric(series.astype(str).str.rstrip('%'), errors='coerce')

# ═══════════════════════════════════════════════════════════════════════════
# STEP 1 ─ Base: API Player Stats  (3 seasons stacked)
# ═══════════════════════════════════════════════════════════════════════════
bases = []
for s in SEASONS:
    sk = s.replace('-', '_')
    bases.append(pd.read_csv(f'{DATA_DIR}/API_SEASON_{sk}/multi_competition_player_stats_{sk}.csv'))

master = pd.concat(bases, ignore_index=True)
master['date'] = to_date(master['date'])
master['_np']  = master['player_name'].map(norm)   # normalised player key
master['_nt']  = master['player_team'].map(norm)   # normalised team key

print(f"Step 1 ─ Base:            {master.shape[0]:,} rows  {master.shape[1]} cols")
print(master['competition'].value_counts().to_string())

# ═══════════════════════════════════════════════════════════════════════════
# STEP 2 ─ JOIN 1: API Team Results   [fixture_id + team]
# Adds: goals_for/against, result, points, is_home,
#       team_shots/possession/corners/fouls, opp_shots/possession
# ═══════════════════════════════════════════════════════════════════════════
tr_list = []
for s in SEASONS:
    sk = s.replace('-', '_')
    tr_list.append(pd.read_csv(f'{DATA_DIR}/API_SEASON_{sk}/multi_competition_team_results_{sk}.csv'))
tr = pd.concat(tr_list, ignore_index=True)

# Convert % string columns → float
for c in ['home_ball_possession', 'away_ball_possession', 'home_passes_%', 'away_passes_%']:
    if c in tr.columns:
        tr[c] = pct_float(tr[c])

# Derive team-perspective stats using is_home flag
for stat, hc, ac in [
    ('shots_on_goal',  'home_shots_on_goal',     'away_shots_on_goal'),
    ('total_shots',    'home_total_shots',        'away_total_shots'),
    ('possession',     'home_ball_possession',    'away_ball_possession'),
    ('corner_kicks',   'home_corner_kicks',       'away_corner_kicks'),
    ('fouls',          'home_fouls',              'away_fouls'),
    ('gk_saves',       'home_goalkeeper_saves',   'away_goalkeeper_saves'),
]:
    if hc in tr.columns and ac in tr.columns:
        tr[f'team_{stat}'] = np.where(tr['is_home'], tr[hc], tr[ac])
        tr[f'opp_{stat}']  = np.where(tr['is_home'], tr[ac], tr[hc])

keep = ['fixture_id', 'team_name', 'is_home', 'opponent',
        'goals_for', 'goals_against', 'result', 'points',
        'team_shots_on_goal', 'team_total_shots', 'team_possession',
        'team_corner_kicks', 'team_fouls', 'team_gk_saves',
        'opp_shots_on_goal', 'opp_total_shots', 'opp_possession']
keep = [c for c in keep if c in tr.columns]

tr_slim = tr[keep].copy()
tr_slim['_nt'] = tr_slim['team_name'].map(norm)
tr_slim = tr_slim.drop(columns='team_name').rename(columns={'opponent': 'opponent_team'})

master = master.merge(tr_slim, on=['fixture_id', '_nt'], how='left')
print(f"\nStep 2 ─ + Team Results:  {master.shape[0]:,} rows  {master.shape[1]} cols  "
      f"| matched {master['goals_for'].notna().sum():,}/{len(master):,}")

# ═══════════════════════════════════════════════════════════════════════════
# STEP 3 ─ JOIN 2: SofaScore Fatigue Metrics   [date + player + team]  PL only
# Adds: rest_days, acwr_ratio, min_last_7d, high_congestion_flag,
#       consecutive_away_games, ss_minutes, sofascore_rating
# ═══════════════════════════════════════════════════════════════════════════
SS_COLS = ['match_date_str', 'name', 'teamName', 'rest_days', 'acwr_ratio',
           'min_last_7d', 'high_congestion_flag', 'consecutive_away_games',
           'minutesPlayed', 'rating']
ss_list = []
for s in SEASONS:
    p = f'{DATA_DIR}/{s}/sofascore_dynamic/fixtureiq_dynamic_master.csv'
    if os.path.exists(p):
        ss_list.append(pd.read_csv(p, usecols=lambda c: c in SS_COLS))

ss = pd.concat(ss_list, ignore_index=True)
ss['match_date_str'] = to_date(ss['match_date_str'])
ss['rest_days']      = ss['rest_days'].clip(upper=21)   # cap outliers (max 294 → 21)
ss = ss.rename(columns={'rating': 'sofascore_rating', 'minutesPlayed': 'ss_minutes'})
ss['_np'] = ss['name'].map(norm)
ss['_nt'] = ss['teamName'].map(norm)
ss = ss.drop(columns=['name', 'teamName'])
ss = ss.drop_duplicates(subset=['match_date_str', '_np', '_nt'])

master = master.merge(ss,
                      left_on=['date', '_np', '_nt'],
                      right_on=['match_date_str', '_np', '_nt'],
                      how='left')
master = master.drop(columns='match_date_str')
print(f"Step 3 ─ + SofaScore:     {master.shape[0]:,} rows  {master.shape[1]} cols  "
      f"| matched {master['sofascore_rating'].notna().sum():,}/{len(master):,}")

# ═══════════════════════════════════════════════════════════════════════════
# STEP 4 ─ JOIN 3: FBRef Action Stats   [date + player + team]  4 teams/season
# Adds: fb_min, fb_shots, fb_sot, fb_tackles_won, fb_crosses,
#       fb_interceptions, fb_fouls, fb_fouled, fb_goals, fb_assists
# ═══════════════════════════════════════════════════════════════════════════
FBREF_TEAMS = {
    '2022-2023': ['chelsea_2022_2023', 'liverpool_2022_2023',
                  'manchester_city_2022_2023', 'tottenham_hotspur_2022_2023'],
    '2023-2024': ['arsenal_2023_2024', 'manchester_city_2023_2024',
                  'manchester_united_2023_2024', 'newcastle_united_2023_2024'],
    '2024-2025': ['arsenal_2024_2025', 'aston_villa_2024_2025',
                  'liverpool_2024_2025', 'manchester_city_2024_2025'],
}
FB_COLS = ['match_date', 'Player', 'Team', 'Min', 'Gls', 'Ast',
           'shots', 'shots_on_target', 'tackles_won', 'crosses',
           'interceptions', 'fouls', 'fouled', 'offsides']

fb_list = []
for s, teams in FBREF_TEAMS.items():
    for t in teams:
        p = f'{DATA_DIR}/{s}/fbref/{t}/match_reports/master_player_stats.csv'
        if os.path.exists(p):
            df = pd.read_csv(p, parse_dates=['match_date'])
            df = df[df['Player'] != 'Matches']                        # drop header artifact rows
            df['Min'] = pd.to_numeric(                                # fix "3,465" → 3465
                df['Min'].astype(str).str.replace(',', '', regex=False), errors='coerce')
            fb_list.append(df[[c for c in FB_COLS if c in df.columns]])

fb = pd.concat(fb_list, ignore_index=True)
fb['match_date'] = to_date(fb['match_date'])
fb = fb.rename(columns={
    'Min': 'fb_min', 'Gls': 'fb_goals', 'Ast': 'fb_assists',
    'shots': 'fb_shots', 'shots_on_target': 'fb_sot',
    'tackles_won': 'fb_tackles_won', 'crosses': 'fb_crosses',
    'interceptions': 'fb_interceptions', 'fouls': 'fb_fouls',
    'fouled': 'fb_fouled', 'offsides': 'fb_offsides',
})
fb['_np'] = fb['Player'].map(norm)
fb['_nt'] = fb['Team'].map(norm)
fb = fb.drop(columns=['Player', 'Team'])
fb = fb.drop_duplicates(subset=['match_date', '_np', '_nt'])

master = master.merge(fb,
                      left_on=['date', '_np', '_nt'],
                      right_on=['match_date', '_np', '_nt'],
                      how='left')
master = master.drop(columns='match_date')
print(f"Step 4 ─ + FBRef:         {master.shape[0]:,} rows  {master.shape[1]} cols  "
      f"| matched {master['fb_min'].notna().sum():,}/{len(master):,}")

# ═══════════════════════════════════════════════════════════════════════════
# STEP 5 ─ JOIN 4: Squad Injury Burden   [date + team]
# Adds: squad_injured_count, squad_soft_tissue_count, squad_avg_days_out
# ═══════════════════════════════════════════════════════════════════════════
it_list = []
for s in SEASONS:
    p = f'{DATA_DIR}/{s}/injuries/team_match_injury_outcomes_{s}.csv'
    if os.path.exists(p):
        it_list.append(pd.read_csv(p))

it = pd.concat(it_list, ignore_index=True)
it['Fixture_Date'] = to_date(it['Fixture_Date'])
it['_nt'] = it['Team'].map(norm)
it = it.rename(columns={
    'Players_Missing_Injury':  'squad_injured_count',
    'Soft_Tissue_Missing':     'squad_soft_tissue_count',
    'Mean_Reported_Days_Out':  'squad_avg_days_out',
})
it_slim = it[['Fixture_Date', '_nt', 'squad_injured_count',
              'squad_soft_tissue_count', 'squad_avg_days_out']].copy()

master = master.merge(it_slim,
                      left_on=['date', '_nt'],
                      right_on=['Fixture_Date', '_nt'],
                      how='left')
master = master.drop(columns='Fixture_Date')
print(f"Step 5 ─ + Squad Injury:  {master.shape[0]:,} rows  {master.shape[1]} cols  "
      f"| matched {master['squad_injured_count'].notna().sum():,}/{len(master):,}")

# ═══════════════════════════════════════════════════════════════════════════
# STEP 6 ─ JOIN 5: Individual Injury Flag   [date + player + team]
# Adds: is_injured (0/1), injury_days_out, injury_type
# Excludes 'Inactive' rows — keeps only real injury absences
# ═══════════════════════════════════════════════════════════════════════════
ia_list = []
for s in SEASONS:
    p = f'{DATA_DIR}/{s}/injuries/ALL_TEAMS_{s}_injuries_days_out.csv'
    if os.path.exists(p):
        ia_list.append(pd.read_csv(p))

ia = pd.concat(ia_list, ignore_index=True)
ia = ia[ia['reason'] == 'Injury'].copy()
ia['fixture_date'] = to_date(ia['fixture_date'])
ia['_np'] = ia['player_name'].map(norm)
ia['_nt'] = ia['team_name'].map(norm)
ia['is_injured'] = 1

# Aggregate: one row per (date, player, team) — take max days_out if multiple injury spells
ia_slim = (ia.groupby(['fixture_date', '_np', '_nt'], as_index=False)
             .agg(is_injured=('is_injured', 'max'),
                  injury_days_out=('days_out', 'max'),
                  injury_type=('injury_type', 'first')))

master = master.merge(ia_slim,
                      left_on=['date', '_np', '_nt'],
                      right_on=['fixture_date', '_np', '_nt'],
                      how='left')
master = master.drop(columns='fixture_date')
master['is_injured'] = master['is_injured'].fillna(0).astype(int)
print(f"Step 6 ─ + Indiv Injury:  {master.shape[0]:,} rows  {master.shape[1]} cols  "
      f"| injured rows: {master['is_injured'].sum():,}")

# ═══════════════════════════════════════════════════════════════════════════
# STEP 7 ─ Target Variable: next-match SofaScore rating per player
# next_sofascore_rating : continuous regression target
# rating_decline_flag   : 1 if next rating drops > 0.5 vs current match
# ═══════════════════════════════════════════════════════════════════════════
master = master.sort_values(['player_id', 'date']).reset_index(drop=True)
master['next_sofascore_rating'] = (
    master.groupby('player_id')['sofascore_rating'].shift(-1)
)
master['rating_decline_flag'] = (
    master['sofascore_rating'].notna() &
    master['next_sofascore_rating'].notna() &
    (master['next_sofascore_rating'] < master['sofascore_rating'] - 0.5)
).astype(int)

# ═══════════════════════════════════════════════════════════════════════════
# STEP 8 ─ Drop temp join keys & Save
# ═══════════════════════════════════════════════════════════════════════════
master = master.drop(columns=['_np', '_nt'])
master.to_csv(OUT_PATH, index=False)

season_col = 'season_x' if 'season_x' in master.columns else 'season'

print(f"\n{'═'*65}")
print(f"  ✅  Fixture_IQ_Data_Seasons.csv  SAVED")
print(f"{'═'*65}")
print(f"  Path : {OUT_PATH}")
print(f"  Size : {os.path.getsize(OUT_PATH)/1e6:.1f} MB")
print(f"  Shape: {master.shape[0]:,} rows × {master.shape[1]} columns")

print(f"\n  Rows by competition:")
print(master['competition'].value_counts().to_string())

print(f"\n  Rows by season:")
print(master[season_col].value_counts().to_string())

print(f"\n  Feature fill-rate (% of rows with data):")
for col, label in [
    ('rating',               'API player rating'),
    ('sofascore_rating',     'SofaScore rating (PL only)'),
    ('rest_days',            'rest_days (fatigue)'),
    ('acwr_ratio',           'ACWR ratio'),
    ('fb_min',               'FBRef action stats'),
    ('squad_injured_count',  'Squad injury burden'),
    ('is_injured',           'Individual injury flag (1s)'),
    ('next_sofascore_rating','TARGET — next match rating'),
    ('rating_decline_flag',  'TARGET — decline flag (1s)'),
]:
    if col in master.columns:
        if col in ('is_injured', 'rating_decline_flag'):
            n   = int(master[col].sum())
            pct = master[col].mean() * 100
        else:
            n   = int(master[col].notna().sum())
            pct = n / len(master) * 100
        print(f"    {label:<38}  {n:7,}  ({pct:.1f}%)")


Step 1 ─ Base:            68,643 rows  36 cols
competition
Premier League      45552
League Cup          10997
FA Cup               7837
Champions League     4137
Community Shield      120

Step 2 ─ + Team Results:  68,643 rows  51 cols  | matched 68,643/68,643
Step 3 ─ + SofaScore:     68,643 rows  58 cols  | matched 12,591/68,643
Step 4 ─ + FBRef:         68,643 rows  69 cols  | matched 7,022/68,643
Step 5 ─ + Squad Injury:  68,643 rows  72 cols  | matched 45,552/68,643
Step 6 ─ + Indiv Injury:  68,643 rows  75 cols  | injured rows: 0

═════════════════════════════════════════════════════════════════
  ✅  Fixture_IQ_Data_Seasons.csv  SAVED
═════════════════════════════════════════════════════════════════
  Path : /tmp/fixture-iq-repo/XgBoost_model/Fixture_IQ_Data_Seasons_2022-2025.csv
  Size : 18.6 MB
  Shape: 68,643 rows × 75 columns

  Rows by competition:
competition
Premier League      45552
League Cup          10997
FA Cup               7837
Champions League     4137
Community S

In [26]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9b — Re-engineer Individual Injury Features (Lag-based)
#
# Why the direct join returned 0 matches:
#   ALL_TEAMS_injuries_days_out records fixtures a player MISSED.
#   The master table only contains fixtures a player PLAYED.
#   These two states are mutually exclusive — so a direct join finds nothing.
#
# Correct approach: for each player-match row, look BACK in the absence
# records and count how many fixtures they missed due to injury in a window.
#
# New features (replace the broken is_injured / injury_days_out / injury_type):
#   fixtures_missed_last_30d  – count of missed fixtures due to injury in last 30 days
#   fixtures_missed_last_90d  – count of missed fixtures due to injury in last 90 days
#   returning_from_injury     – 1 if player missed ≥1 fixture in the last 30 days
#   days_since_last_injury    – days since most recent injury absence (NaN if none)
# ─────────────────────────────────────────────────────────────────────────────

from bisect import bisect_left, bisect_right
import unicodedata, os
import numpy as np
import pandas as pd

DATA_DIR = '/tmp/fixture-iq-repo/XgBoost_model/data'
SEASONS  = ['2022-2023', '2023-2024', '2024-2025']
OUT_PATH = '/tmp/fixture-iq-repo/XgBoost_model/Fixture_IQ_Data_Seasons_2022-2025.csv'

def norm(s):
    if pd.isna(s): return ''
    s = unicodedata.normalize('NFD', str(s).lower().strip())
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.replace("'", '').replace('-', ' ').replace('.', '').strip()

def to_date(series):
    return pd.to_datetime(series, errors='coerce', utc=True).dt.tz_convert(None).dt.normalize()

# ── 1. Load & prepare all injury absence records ─────────────────────────────
ia_list = []
for s in SEASONS:
    p = f'{DATA_DIR}/{s}/injuries/ALL_TEAMS_{s}_injuries_days_out.csv'
    if os.path.exists(p):
        ia_list.append(pd.read_csv(p))

ia = pd.concat(ia_list, ignore_index=True)

# Exclude non-physical absence reasons (suspensions, admin, international duty, etc.)
# Everything else is a physical injury / illness / surgery worth counting
EXCLUDE_REASONS = {
    'Suspended', 'Yellow Cards', 'Red Card',
    'Inactive', 'Lacking Match Fitness', 'Rest',
    'Loan agreement', "Coach's decision", 'Personal Reasons',
    'International duty', 'National selection',
}
ia = ia[~ia['reason'].isin(EXCLUDE_REASONS)].copy()
ia['fixture_date'] = to_date(ia['fixture_date'])
ia['_np'] = ia['player_name'].map(norm)
ia['_nt'] = ia['team_name'].map(norm)
ia = ia.dropna(subset=['fixture_date'])
ia = ia.drop_duplicates(subset=['_np', '_nt', 'fixture_date'])

print(f"Physical absence records (injuries / illness / surgery): {len(ia):,}")
print(ia['reason'].value_counts().head(12).to_string())

# ── 2. Build fast lookup: (norm_player, norm_team) → sorted list of dates ────
# Binary search on sorted dates makes each row lookup O(log n) instead of O(n)
injury_lookup = {}
for (np_, nt_), grp in ia.groupby(['_np', '_nt']):
    injury_lookup[(np_, nt_)] = sorted(grp['fixture_date'].tolist())

print(f"\nUnique (player, team) pairs with injury history: {len(injury_lookup):,}")

# ── 3. Load master & compute lag features for every row ──────────────────────
master = pd.read_csv(OUT_PATH)
master['date'] = pd.to_datetime(master['date'])
master['_np']  = master['player_name'].map(norm)
master['_nt']  = master['player_team'].map(norm)

print(f"\nComputing lag injury features for {len(master):,} rows...")

missed_30  = np.zeros(len(master), dtype=np.int16)
missed_90  = np.zeros(len(master), dtype=np.int16)
days_since = np.full(len(master), np.nan, dtype=float)

for i, (d, np_, nt_) in enumerate(zip(master['date'].tolist(),
                                       master['_np'].tolist(),
                                       master['_nt'].tolist())):
    dates = injury_lookup.get((np_, nt_))
    if dates is None:
        continue

    # Upper bound: all injury dates strictly before match day
    hi = bisect_left(dates, d)

    # 30-day window: [d - 30 days, d - 1 day]
    lo30 = bisect_left(dates, d - pd.Timedelta(days=30))
    missed_30[i] = max(0, hi - lo30)

    # 90-day window: [d - 90 days, d - 1 day]
    lo90 = bisect_left(dates, d - pd.Timedelta(days=90))
    missed_90[i] = max(0, hi - lo90)

    # Days since the most recent injury absence before this match
    last_idx = hi - 1
    if last_idx >= 0:
        days_since[i] = (d - dates[last_idx]).days

master['fixtures_missed_last_30d'] = missed_30
master['fixtures_missed_last_90d'] = missed_90
master['returning_from_injury']    = (missed_30 > 0).astype(np.int8)
master['days_since_last_injury']   = days_since

# ── 4. Drop the old broken zero-columns, clean up, re-save ───────────────────
master = master.drop(columns=['is_injured', 'injury_days_out', 'injury_type',
                               '_np', '_nt'], errors='ignore')
master.to_csv(OUT_PATH, index=False)

print(f"\n{'═'*65}")
print(f"  ✅  Lag injury features computed — master table updated")
print(f"{'═'*65}")
print(f"  Path : {OUT_PATH}")
print(f"  Shape: {master.shape[0]:,} rows × {master.shape[1]} columns")

print(f"\n  New injury feature summary:")
print(f"  {'Feature':<35}  {'non-zero / non-null':>10}  {'rate':>8}")
print(f"  {'─'*60}")
for col, label in [
    ('returning_from_injury',    'returning_from_injury'),
    ('fixtures_missed_last_30d', 'fixtures_missed_last_30d (>0)'),
    ('fixtures_missed_last_90d', 'fixtures_missed_last_90d (>0)'),
    ('days_since_last_injury',   'days_since_last_injury (non-null)'),
]:
    if col == 'days_since_last_injury':
        n = int(master[col].notna().sum()); pct = n/len(master)*100
    else:
        n = int((master[col] > 0).sum()); pct = n/len(master)*100
    print(f"  {label:<35}  {n:>10,}  {pct:>7.1f}%")

print(f"\n  fixtures_missed_last_30d value counts (top 5):")
print(master['fixtures_missed_last_30d'].value_counts().sort_index().head(5).to_string())

print(f"\n  days_since_last_injury — stats (non-null rows only):")
print(master['days_since_last_injury'].dropna().describe().round(1).to_string())


Physical absence records (injuries / illness / surgery): 8,512
reason
Knee Injury               2232
Thigh Injury              1421
Muscle Injury              924
Ankle Injury               908
Injury                     653
Calf Injury                524
Hamstring Injury           232
Groin Injury               206
Knock                      169
Shoulder Injury            156
Achilles Tendon Injury     152
Foot Injury                137

Unique (player, team) pairs with injury history: 680

Computing lag injury features for 68,643 rows...

═════════════════════════════════════════════════════════════════
  ✅  Lag injury features computed — master table updated
═════════════════════════════════════════════════════════════════
  Path : /tmp/fixture-iq-repo/XgBoost_model/Fixture_IQ_Data_Seasons_2022-2025.csv
  Shape: 68,643 rows × 76 columns

  New injury feature summary:
  Feature                              non-zero / non-null      rate
  ──────────────────────────────────────────────

## Section 9c — Recompute Workload & Fatigue Features for ALL Rows

The five fatigue columns inherited from the SofaScore join (`rest_days`, `min_last_7d`, `acwr_ratio`, `high_congestion_flag`, `consecutive_away_games`) had **74.6% nulls** because SofaScore only covered Premier League fixtures.

Every input needed to compute these features is already in the master table for every row:
- `date` → rest days and rolling windows
- `minutes_played` → acute and chronic workload
- `is_home` → consecutive away game streaks

The cell below recomputes all five features for all 68,643 rows using a per-player sorted timeline and binary-search window lookups (O(N log N) total). The sparse SofaScore columns are replaced in-place with full-coverage equivalents.

**Limitation**: for a player's first match in the dataset (2022-23 start), `rest_days` is NaN and rolling windows start from zero — there is no pre-2022 history available to look back into.


In [27]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9c — Compute Workload & Fatigue Features for ALL Rows
#
# Inputs already in master (0% null):
#   date           → rest_days, rolling minute windows
#   minutes_played → min_last_7d, min_last_28d, acwr_ratio
#   is_home        → consecutive_away_games
#
# Features computed:
#   rest_days              – days since player's previous fixture (clipped ≤ 21;
#                            NaN for first appearance in dataset)
#   min_last_7d            – minutes played in [d−7d, d−1d] window
#   acwr_ratio             – min_last_7d / (min_last_28d / 4)  [acute÷chronic]
#   high_congestion_flag   – 1 if rest_days ≤ 6 (≥2 matches in a 7-day window)
#   consecutive_away_games – count of consecutive away matches up to & including this one
#   ss_minutes             – null-filled from minutes_played where SofaScore was absent
#
# All five columns are overwritten in the master table (SofaScore versions replaced).
# ─────────────────────────────────────────────────────────────────────────────

from bisect import bisect_left
import unicodedata
import numpy as np
import pandas as pd

OUT_PATH = '/tmp/fixture-iq-repo/XgBoost_model/Fixture_IQ_Data_Seasons_2022-2025.csv'

def norm(s):
    if pd.isna(s): return ''
    s = unicodedata.normalize('NFD', str(s).lower().strip())
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.replace("'", '').replace('-', ' ').replace('.', '').strip()

# ── 1. Load master ────────────────────────────────────────────────────────────
master = pd.read_csv(OUT_PATH)
master['date'] = pd.to_datetime(master['date'])

# Group key: normalised (player_name, player_team)
# Using player_id alone risks collapsing different players who share player_id=0
master['_gk'] = master['player_name'].map(norm) + '||' + master['player_team'].map(norm)

# Must sort by (player group, date) before building per-player timelines
master = master.sort_values(['_gk', 'date']).reset_index(drop=True)

n = len(master)
rest_days_arr    = np.full(n, np.nan, dtype=float)
min_last_7d_arr  = np.zeros(n, dtype=float)
min_last_28d_arr = np.zeros(n, dtype=float)
consec_away_arr  = np.zeros(n, dtype=np.int16)

n_players = master['_gk'].nunique()
print(f"Computing workload features for {n_players:,} player-team combinations ({n:,} rows)...")

# ── 2. Per-player group computation ──────────────────────────────────────────
for gk, grp_idx in master.groupby('_gk', sort=False).groups.items():
    idx     = grp_idx.tolist()          # integer positions, already in date order
    dates   = master.loc[idx, 'date'].tolist()
    mins    = master.loc[idx, 'minutes_played'].tolist()
    is_away = (~master.loc[idx, 'is_home'].astype(bool)).tolist()
    ng      = len(idx)

    # Prefix sum of minutes — turns any range sum into two O(1) lookups
    pfx = [0.0] * (ng + 1)
    for j, m in enumerate(mins):
        pfx[j + 1] = pfx[j] + m

    # Consecutive away streak: forward pass, reset on every home game
    consec = [0] * ng
    consec[0] = 1 if is_away[0] else 0
    for j in range(1, ng):
        consec[j] = (consec[j - 1] + 1) if is_away[j] else 0

    for i, gi in enumerate(idx):
        d = dates[i]

        # rest_days: gap to previous match, clipped to 21
        if i > 0:
            rest_days_arr[gi] = min(21.0, float((d - dates[i - 1]).days))

        # Rolling window indices (all fixtures strictly before match day d)
        hi   = bisect_left(dates, d)                        # == i for unique dates
        lo7  = bisect_left(dates, d - pd.Timedelta(days=7))
        lo28 = bisect_left(dates, d - pd.Timedelta(days=28))

        min_last_7d_arr[gi]  = pfx[hi] - pfx[lo7]
        min_last_28d_arr[gi] = pfx[hi] - pfx[lo28]

        consec_away_arr[gi] = consec[i]

# ── 3. Derive acwr_ratio and high_congestion_flag ────────────────────────────
# Chronic load = average weekly minutes over past 28 days
chronic  = min_last_28d_arr / 4.0
acwr     = np.where(chronic > 0, min_last_7d_arr / chronic, 0.0).clip(max=4.0)

# Congestion: previous match was within 6 days → 2+ matches in a 7-day window
# NaN rest_days (first appearance) → 0
high_cong = np.where(np.isnan(rest_days_arr), 0.0,
                     (rest_days_arr <= 6).astype(float))

# ── 4. Write back — overwrite the sparse SofaScore versions ──────────────────
master['rest_days']              = rest_days_arr
master['min_last_7d']            = min_last_7d_arr
master['acwr_ratio']             = np.round(acwr, 3)
master['high_congestion_flag']   = high_cong
master['consecutive_away_games'] = consec_away_arr.astype(float)

# ss_minutes: fill any remaining nulls with minutes_played (same stat, different source)
master['ss_minutes'] = master['ss_minutes'].fillna(master['minutes_played'])

# ── 5. Clean up, restore sort, re-save ───────────────────────────────────────
master = master.drop(columns='_gk')
master = master.sort_values(['player_id', 'date']).reset_index(drop=True)
master.to_csv(OUT_PATH, index=False)

# ── 6. Report ─────────────────────────────────────────────────────────────────
print(f"\n{'═'*68}")
print(f"  ✅  Workload features recomputed — full coverage achieved")
print(f"{'═'*68}")
print(f"  Path : {OUT_PATH}")
print(f"  Shape: {master.shape[0]:,} rows × {master.shape[1]} columns")

print(f"\n  {'Feature':<28}  {'Null% before':>13}  {'Null% after':>12}  {'Mean':>8}  {'Max':>7}")
print(f"  {'─'*74}")
for col, before in [
    ('rest_days',              74.6),
    ('min_last_7d',            74.6),
    ('acwr_ratio',             74.6),
    ('high_congestion_flag',   74.6),
    ('consecutive_away_games', 74.6),
    ('ss_minutes',             74.6),
]:
    null_now = master[col].isna().mean() * 100
    mean_val = master[col].mean()
    max_val  = master[col].max()
    print(f"  {col:<28}  {before:>12.1f}%  {null_now:>11.1f}%  {mean_val:>8.2f}  {max_val:>7.1f}")

print(f"\n  rest_days (days between consecutive appearances):")
print(master['rest_days'].dropna().describe().round(1).to_string())

print(f"\n  acwr_ratio (acute÷chronic workload):")
print(master['acwr_ratio'].describe().round(3).to_string())

print(f"\n  high_congestion_flag: "
      f"{master['high_congestion_flag'].mean()*100:.1f}% of appearances "
      f"({int(master['high_congestion_flag'].sum()):,} rows)")
print(f"  consecutive_away_games max: {int(master['consecutive_away_games'].max())}  "
      f"mean: {master['consecutive_away_games'].mean():.2f}")


Computing workload features for 6,895 player-team combinations (68,643 rows)...


/tmp/ipykernel_1669970/1716269607.py:93: RuntimeWarning: invalid value encountered in divide
  acwr     = np.where(chronic > 0, min_last_7d_arr / chronic, 0.0).clip(max=4.0)



════════════════════════════════════════════════════════════════════
  ✅  Workload features recomputed — full coverage achieved
════════════════════════════════════════════════════════════════════
  Path : /tmp/fixture-iq-repo/XgBoost_model/Fixture_IQ_Data_Seasons_2022-2025.csv
  Shape: 68,643 rows × 76 columns

  Feature                        Null% before   Null% after      Mean      Max
  ──────────────────────────────────────────────────────────────────────────
  rest_days                             74.6%         10.0%      8.88     21.0
  min_last_7d                           74.6%          0.0%     34.78    215.0
  acwr_ratio                            74.6%          0.0%      0.75      4.0
  high_congestion_flag                  74.6%          0.0%      0.41      1.0
  consecutive_away_games                74.6%          0.0%      0.68      7.0
  ss_minutes                            74.6%          0.0%     49.86    120.0

  rest_days (days between consecutive appearances):
co

## Section 9d — Recompute Target Variables Using API Rating

### Problem with the SofaScore-based targets

| Target | Coverage | Positive class | Issue |
|--------|----------|----------------|-------|
| `next_sofascore_rating` | **18.3%** non-null | — | SofaScore = Premier League only |
| `rating_decline_flag` | 100% | **2.3%** | Non-PL rows forced to 0 → artificially inflated negative class; `scale_pos_weight ≈ 41.8` |

### Solution: API `rating` column (100% rows, all competitions)

The API `rating` column covers every player in every competition across all 3 seasons.  
`rating = 0` is a sentinel (player came on for <15 min, GK not tested, etc.) and is treated as null.

| Threshold | `next_api_rating` fill | Positive% | `scale_pos_weight` |
|-----------|----------------------|-----------|-------------------|
| 0.30 | 67.3% | 18.9% | ~4.3 |
| **0.50** | **67.3%** | **13.3%** | **~6.5** ← default |
| 0.75 | 67.3% | 8.2% | ~11.2 |
| 1.00 | 67.3% | 4.3% | ~22.3 |

**Fix also applied**: group key changed from `player_id` (broken — thousands of players share `player_id = 0`) to normalised `(player_name + player_team)`.

New columns added to master table:
- `next_api_rating` — regression target (API-based)
- `api_rating_decline_flag` — classification target (API-based, threshold = 0.5)

In [28]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9d — Recompute Target Variables Using API Rating
#
# Why SofaScore targets fail:
#   next_sofascore_rating  → 82.3% null  (SofaScore = Premier League only)
#   rating_decline_flag    → 2.3% positive class only; non-PL rows all forced to 0
#                            → scale_pos_weight ≈ 41.8 in XGBoost (extremely imbalanced)
#   player_id grouping     → player_id = 0 is shared by thousands of players
#                            → shift(-1) contaminates targets across unrelated players
#
# Solution: API `rating` column
#   ✅ 0% null — present for all 68,643 rows
#   ✅ All competitions: PL, CL, FA Cup, League Cup
#   ⚠ rating = 0 means no rating was assigned (sub <15 min, GK not tested, etc.)
#      → treat rating = 0 as null for target computation
#
# Group key fix: normalised (player_name + player_team) instead of player_id
#
# New columns:
#   next_api_rating        – API rating in player's next match  [regression target]
#   api_rating_decline_flag – 1 if next_api_rating < current − DECLINE_THRESHOLD
#                                                               [classification target]
# ─────────────────────────────────────────────────────────────────────────────

import unicodedata, os
import numpy as np
import pandas as pd

OUT_PATH         = '/tmp/fixture-iq-repo/XgBoost_model/Fixture_IQ_Data_Seasons_2022-2025.csv'
DECLINE_THRESHOLD = 0.5   # points drop to call a "decline" — adjust to taste

def norm(s):
    if pd.isna(s): return ''
    s = unicodedata.normalize('NFD', str(s).lower().strip())
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.replace("'", '').replace('-', ' ').replace('.', '').strip()

# ── 1. Load master ─────────────────────────────────────────────────────────
master = pd.read_csv(OUT_PATH)
master['date'] = pd.to_datetime(master['date'])

print(f"Loaded: {master.shape[0]:,} rows × {master.shape[1]} cols")
print(f"\nAPI rating distribution:")
print(f"  rating = 0  (no rating assigned): {(master['rating']==0).sum():,}  "
      f"({(master['rating']==0).mean()*100:.1f}%)")
print(f"  rating > 0  (real rating):         {(master['rating']>0).sum():,}  "
      f"({(master['rating']>0).mean()*100:.1f}%)")
print(f"  rating nulls:                       {master['rating'].isna().sum()}")

# ── 2. Proper group key — avoids player_id = 0 collision ──────────────────
master['_gk'] = master['player_name'].map(norm) + '||' + master['player_team'].map(norm)
n_groups = master['_gk'].nunique()
print(f"\nUnique (player, team) pairs: {n_groups:,}  (vs player_id unique: {master['player_id'].nunique():,})")

# ── 3. Treat rating = 0 as null (no meaningful rating assigned) ───────────
master['_api_r'] = master['rating'].replace(0, np.nan)

# ── 4. Sort by (group, date) and compute next_api_rating via shift(-1) ────
master = master.sort_values(['_gk', 'date']).reset_index(drop=True)
master['next_api_rating'] = master.groupby('_gk')['_api_r'].shift(-1)

# ── 5. Compute api_rating_decline_flag ─────────────────────────────────────
# Flag = 1 only when BOTH current and next ratings are real (non-null, non-zero)
master['api_rating_decline_flag'] = (
    master['_api_r'].notna() &
    master['next_api_rating'].notna() &
    (master['next_api_rating'] < master['_api_r'] - DECLINE_THRESHOLD)
).astype(int)

# ── 6. Drop temp columns, restore player_id sort, re-save ─────────────────
master = master.drop(columns=['_gk', '_api_r'])
master = master.sort_values(['player_id', 'date']).reset_index(drop=True)
master.to_csv(OUT_PATH, index=False)

# ── 7. Full comparison report ──────────────────────────────────────────────
print(f"\n{'═'*72}")
print(f"  ✅  API-based targets added — master table updated")
print(f"{'═'*72}")
print(f"  Path : {OUT_PATH}")
print(f"  Shape: {master.shape[0]:,} rows × {master.shape[1]} columns")

print(f"\n  {'─'*70}")
print(f"  TARGET VARIABLE COMPARISON")
print(f"  {'─'*70}")
print(f"  {'Target':<35}  {'Non-null':>9}  {'Fill%':>7}  {'Positives':>10}  {'Pos%':>7}  {'Impl.weight':>12}")
print(f"  {'─'*70}")

def report_target(col, label, is_flag=False):
    if col not in master.columns:
        return
    nn  = master[col].notna().sum()
    pct = nn / len(master) * 100
    if is_flag:
        pos     = int(master[col].sum())
        pos_pct = master[col].mean() * 100
        weight  = (1 - master[col].mean()) / master[col].mean() if master[col].mean() > 0 else float('inf')
        print(f"  {label:<35}  {nn:>9,}  {pct:>6.1f}%  {pos:>10,}  {pos_pct:>6.1f}%  {weight:>12.1f}")
    else:
        print(f"  {label:<35}  {nn:>9,}  {pct:>6.1f}%  {'—':>10}  {'—':>7}  {'—':>12}")

# Old SofaScore targets
report_target('next_sofascore_rating', 'next_sofascore_rating  [OLD]')
report_target('rating_decline_flag',   'rating_decline_flag    [OLD]', is_flag=True)
print(f"  {'·'*70}")
# New API targets
report_target('next_api_rating',        'next_api_rating        [NEW]')
report_target('api_rating_decline_flag','api_rating_decline_flag[NEW]', is_flag=True)

print(f"\n  Threshold used: {DECLINE_THRESHOLD} points  "
      f"(change DECLINE_THRESHOLD above and re-run to explore alternatives)")

print(f"\n  Threshold sensitivity (api_rating_decline_flag):")
print(f"  {'─'*50}")
print(f"  {'Threshold':>10}  {'Positives':>10}  {'Pos%':>7}  {'scale_pos_weight':>18}")
valid_mask = (master['rating'] > 0) & master['next_api_rating'].notna()
for t in [0.30, 0.50, 0.75, 1.00]:
    flag_t = valid_mask & (master['next_api_rating'] < master['rating'] - t)
    pos    = flag_t.sum()
    pp     = flag_t.mean() * 100
    spw    = (1 - flag_t.mean()) / flag_t.mean() if flag_t.mean() > 0 else float('inf')
    marker = '  ← active' if abs(t - DECLINE_THRESHOLD) < 0.01 else ''
    print(f"  {t:>10.2f}  {pos:>10,}  {pp:>6.1f}%  {spw:>18.1f}{marker}")

print(f"\n  next_api_rating distribution (non-null, non-zero rows):")
print(master[master['next_api_rating'].notna()]['next_api_rating'].describe().round(3).to_string())

print(f"\n  Cross-validation — PL rows where BOTH flags available:")
pl = master[(master['competition'] == 'Premier League') &
            master['rating_decline_flag'].notna() &
            master['next_sofascore_rating'].notna()].copy()
if len(pl) > 0:
    agree = (pl['rating_decline_flag'] == pl['api_rating_decline_flag']).mean() * 100
    print(f"  PL rows with both: {len(pl):,}  |  agreement: {agree:.1f}%")


Loaded: 68,643 rows × 76 cols

API rating distribution:
  rating = 0  (no rating assigned): 17,731  (25.8%)
  rating > 0  (real rating):         50,912  (74.2%)
  rating nulls:                       0

Unique (player, team) pairs: 6,895  (vs player_id unique: 5,386)

════════════════════════════════════════════════════════════════════════
  ✅  API-based targets added — master table updated
════════════════════════════════════════════════════════════════════════
  Path : /tmp/fixture-iq-repo/XgBoost_model/Fixture_IQ_Data_Seasons_2022-2025.csv
  Shape: 68,643 rows × 78 columns

  ──────────────────────────────────────────────────────────────────────
  TARGET VARIABLE COMPARISON
  ──────────────────────────────────────────────────────────────────────
  Target                                Non-null    Fill%   Positives     Pos%   Impl.weight
  ──────────────────────────────────────────────────────────────────────
  next_sofascore_rating  [OLD]            12,148    17.7%           —       

## Section 9e — Starter vs Substitute Bias in Target Variables

### The problem

The `api_rating_decline_flag` is **confounded by role changes**. Substitutes receive a structurally lower rating than starters because they play fewer minutes:

| Role | Mean rating | Std | Mean minutes |
|------|------------|-----|-------------|
| Starter (`is_substitute=0`) | **6.99** | 0.61 | 82.9 min |
| Substitute (`is_substitute=1`) | **6.66** | 0.39 | 9.5 min |

When the *next* match has a different role, the flag is measuring **role demotion**, not pure performance decline:

| Transition (current→next) | Rows | Decline flags | Decline rate |
|---|---|---|---|
| Starter → Starter | 26,051 | 6,540 | 25.1% — genuine signal |
| Starter → Sub | 4,778 | 1,511 | **31.6%** — inflated: fewer minutes = lower rating by construction |
| Sub → Sub | 4,140 | 573 | 13.8% — noisy: two short appearances |
| Sub → Starter | 5,132 | 519 | 10.1% — mostly improvements, minority mislabelled as decline |

### Fix applied in this section

1. **`api_rating_decline_flag` updated** — only fires when the player played ≥ `MIN_MINUTES` (default 45) in **both** the current and next match. This compares like-for-like meaningful appearances only.
2. **Two new context features added** — `next_is_substitute` and `next_minutes_played` so the model can learn role-change patterns even where the flag is suppressed.

In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9e — Starter vs Substitute Bias Correction
#
# Problem: API rating is structurally lower for substitutes (fewer minutes played).
# Comparing a starter's current rating to a substitute's next-match rating
# labels a role demotion as a performance decline — which is misleading.
#
# Fix 1: api_rating_decline_flag — only fire when player played >= MIN_MINUTES
#         in BOTH the current AND the next match.
#
# Fix 2: Add next_is_substitute + next_minutes_played as explicit features so
#         the model can learn role-change patterns independently.
# ─────────────────────────────────────────────────────────────────────────────

import unicodedata, os
import numpy as np
import pandas as pd

OUT_PATH    = '/tmp/fixture-iq-repo/XgBoost_model/Fixture_IQ_Data_Seasons_2022-2025.csv'
MIN_MINUTES = 45   # minimum minutes in BOTH matches to count a rating comparison
DECLINE_THRESHOLD = 0.5

def norm(s):
    if pd.isna(s): return ''
    s = unicodedata.normalize('NFD', str(s).lower().strip())
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    return s.replace("'", '').replace('-', ' ').replace('.', '').strip()

# ── 1. Load & sort ────────────────────────────────────────────────────────
master = pd.read_csv(OUT_PATH)
master['date'] = pd.to_datetime(master['date'])
master['_gk']  = master['player_name'].map(norm) + '||' + master['player_team'].map(norm)
master = master.sort_values(['_gk', 'date']).reset_index(drop=True)

# ── 2. Add next-match context features ───────────────────────────────────
# These let the model learn about role changes even where the flag is suppressed
master['next_is_substitute']  = master.groupby('_gk')['is_substitute'].shift(-1)
master['next_minutes_played'] = master.groupby('_gk')['minutes_played'].shift(-1)

# ── 3. Show rating distribution by role (confirms structural bias) ────────
r = master[master['rating'] > 0]
print("API Rating by role (rating > 0 rows)")
print("─" * 55)
stats = r.groupby('is_substitute')['rating'].agg(['count','mean','std','min','max'])
stats.index = ['Starter (is_substitute=0)', 'Substitute (is_substitute=1)']
print(stats.round(3).to_string())

print("\nMinutes played by role (all rows)")
print("─" * 55)
mstats = master.groupby('is_substitute')['minutes_played'].agg(['count','mean','std','min','max'])
mstats.index = ['Starter', 'Substitute']
print(mstats.round(1).to_string())

# ── 4. Before/after analysis of decline flags by role transition ──────────
print("\nDecline flags by role transition — BEFORE fix")
print("─" * 65)
has_both = master['next_api_rating'].notna() & (master['rating'] > 0)
trans_df = master[has_both].copy()
trans_df['next_sub_int'] = trans_df['next_is_substitute'].fillna(-1).astype(int)
trans_df['transition'] = (
    trans_df['is_substitute'].astype(int).astype(str) + '→' +
    trans_df['next_sub_int'].astype(str)
).map({'0→0': 'Starter→Starter', '1→0': 'Sub→Starter',
       '0→1': 'Starter→Sub',    '1→1': 'Sub→Sub'}).fillna('unknown')

grp = trans_df.groupby('transition').agg(
    rows=('api_rating_decline_flag','count'),
    declines=('api_rating_decline_flag','sum')).assign(
    decline_rate=lambda x: x['declines']/x['rows']*100)
print(grp.to_string())

# ── 5. Update api_rating_decline_flag with minutes guard ─────────────────
# Both current and next match must have >= MIN_MINUTES
current_played_enough = master['minutes_played'] >= MIN_MINUTES
next_played_enough    = master['next_minutes_played'].fillna(0) >= MIN_MINUTES
both_rated            = (master['rating'] > 0) & master['next_api_rating'].notna()

old_flag = master['api_rating_decline_flag'].copy()

master['api_rating_decline_flag'] = (
    current_played_enough &
    next_played_enough    &
    both_rated            &
    (master['next_api_rating'] < master['rating'] - DECLINE_THRESHOLD)
).astype(int)

# ── 6. After analysis ─────────────────────────────────────────────────────
print("\nDecline flags by role transition — AFTER fix (both >= 45 min)")
print("─" * 65)
grp_after = trans_df.copy()
grp_after['api_rating_decline_flag'] = master.loc[trans_df.index, 'api_rating_decline_flag']
grp2 = grp_after.groupby('transition').agg(
    rows=('api_rating_decline_flag','count'),
    declines=('api_rating_decline_flag','sum')).assign(
    decline_rate=lambda x: x['declines']/x['rows']*100)
print(grp2.to_string())

# ── 7. Summary: flag change impact ────────────────────────────────────────
print(f"\n{'═'*68}")
print(f"  Flag change impact (threshold = {DECLINE_THRESHOLD}, min_minutes = {MIN_MINUTES})")
print(f"  {'─'*66}")
removed   = int(((old_flag == 1) & (master['api_rating_decline_flag'] == 0)).sum())
unchanged = int(((old_flag == 1) & (master['api_rating_decline_flag'] == 1)).sum())
print(f"  Flags retained  (genuine min-controlled declines): {unchanged:,}")
print(f"  Flags removed   (role-change / short-appearance):  {removed:,}")
print(f"  New positive rate: {master['api_rating_decline_flag'].mean()*100:.1f}%  "
      f"(was {old_flag.mean()*100:.1f}%)")
new_spw = (1 - master['api_rating_decline_flag'].mean()) / master['api_rating_decline_flag'].mean()
print(f"  New scale_pos_weight: {new_spw:.1f}")

# ── 8. Drop temp key, restore sort, save ─────────────────────────────────
master = master.drop(columns='_gk')
master = master.sort_values(['player_id', 'date']).reset_index(drop=True)
master.to_csv(OUT_PATH, index=False)

print(f"\n  ✅  Master table updated")
print(f"  Path : {OUT_PATH}")
print(f"  Shape: {master.shape[0]:,} rows × {master.shape[1]} columns")
print(f"  New columns: next_is_substitute, next_minutes_played")
print(f"  Updated:     api_rating_decline_flag (now min-guarded)")


API Rating by role (rating > 0 rows)
───────────────────────────────────────────────────────
                              count   mean    std  min   max
Starter (is_substitute=0)     37715  6.989  0.611  3.0  10.0
Substitute (is_substitute=1)  13197  6.661  0.389  3.6   9.5

Minutes played by role (all rows)
───────────────────────────────────────────────────────
            count  mean   std  min  max
Starter     37730  82.9  13.4    4  120
Substitute  30913   9.5  13.8    0   92

Decline flags by role transition — BEFORE fix
─────────────────────────────────────────────────────────────────
                  rows  declines  decline_rate
transition                                    
Starter→Starter  26051      6540     25.104603
Starter→Sub       4778      1511     31.624111
Sub→Starter       5132       519     10.113016
Sub→Sub           4140       573     13.840580

Decline flags by role transition — AFTER fix (both >= 45 min)
───────────────────────────────────────────────────────

/tmp/ipykernel_1669970/2171408987.py:59: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  trans_df['next_sub_int'] = trans_df['next_is_substitute'].fillna(-1).astype(int)



  ✅  Master table updated
  Path : /tmp/fixture-iq-repo/XgBoost_model/Fixture_IQ_Data_Seasons_2022-2025.csv
  Shape: 68,643 rows × 80 columns
  New columns: next_is_substitute, next_minutes_played
  Updated:     api_rating_decline_flag (now min-guarded)


## Section 10 — Master Table: Complete Column Reference
### `Fixture_IQ_Data_Seasons_2022-2025.csv` — 68,643 rows × 80 columns

Each row is **one player in one match**. There are no aggregate rows — every entry is a single appearance.
Columns are grouped by where the data came from and what it represents.

---

## GROUP 1 — Match Identity (cols 1–8)
*What match is this row about?*

| # | Column | Type | Null% | Example | Meaning |
|---|--------|------|-------|---------|---------|
| 1 | `fixture_id` | int | 0% | 1066064 | Unique numeric ID assigned by the API to each individual match. Use this as the primary key when joining to other match-level tables. |
| 2 | `date` | str (ISO date) | 0% | 2024-08-13 | Calendar date the match was played (UTC midnight). Used to compute rest windows, rolling workloads, and time-based train/test splits. |
| 3 | `competition` | str | 0% | Premier League | Competition name: `Premier League`, `Champions League`, `FA Cup`, `League Cup`, `Community Shield` (120 rows). All five are present — this is the key distinguishing column for multi-competition filtering. |
| 4 | `season` | int | 0% | 2024 | Start year of the season (2022 → 2022-23, 2023 → 2023-24, 2024 → 2024-25). Useful for time-based cross-validation. |
| 5 | `round` | str | 0% | 3rd Round | Competition round label (e.g. `Matchweek 1`, `Round of 16`, `Semi-Final`). Highly variable format across competitions — encode carefully or drop. |
| 6 | `home_team` | str | 0% | Manchester City | Full name of the home team in this fixture. |
| 7 | `away_team` | str | 0% | Arsenal | Full name of the away team in this fixture. |
| 8 | `player_team` | str | 0% | Arsenal | Team this player appeared for. Redundant with `home_team`/`away_team` but more convenient for player-centric queries. |

---

## GROUP 2 — Player Identity (cols 9–12)
*Who is this player?*

| # | Column | Type | Null% | Example | Meaning |
|---|--------|------|-------|---------|---------|
| 9 | `player_id` | int | 0% | 504648 | Numeric ID from the API. **Important caveat**: many unmatched players share `player_id = 0`. Do not use alone as a grouping key — always combine with `player_name + player_team`. |
| 10 | `player_name` | str | 0% | Bernardo Silva | Player's display name as returned by the API. Accents and special characters are present (e.g. `Vinícius Júnior`). |
| 11 | `player_number` | int | 0% | 20 | Squad shirt number in this specific match. Can change across seasons. Range: 1–99. |
| 12 | `player_position` | str | 0% | M | Broad position code: **G** (Goalkeeper), **D** (Defender), **M** (Midfielder), **F** (Forward). Source is the API match lineup. |

---

## GROUP 3 — Individual In-Match Performance (cols 13–34)
*What did this specific player do during the match?*
**Source**: API multi-competition player stats. **Coverage**: 100% (all rows).

| # | Column | Type | Null% | Range | Meaning |
|---|--------|------|-------|-------|---------|
| 13 | `minutes_played` | int | 0% | 0–120 | Minutes the player was on the pitch. Includes extra time. 0 = named in squad but did not play. 120 = played full 90 mins + 30 mins extra time. |
| 14 | `rating` | float | 0% | 0–10 | API match rating for the player. Scale 0–10; **0 = no rating assigned** (came on for <15 min, GK not tested, etc.) — treat 0 as null when using as a feature or target. 74.2% of rows have `rating > 0`. |
| 15 | `is_captain` | bool | 0% | 0/1 | 1 if the player wore the captain's armband in this match (~5% of appearances). |
| 16 | `is_substitute` | bool | 0% | 0/1 | 1 if the player entered as a substitute (came on after kickoff). ~45% of appearances. Starters average **6.99** rating; substitutes average **6.66** — structurally lower due to fewer minutes. |
| 17 | `shots_total` | int | 0% | 0–9 | Total shot attempts by this player (on or off target). |
| 18 | `shots_on_target` | int | 0% | 0–8 | Shots that required the goalkeeper to make a save or resulted in a goal. |
| 19 | `goals` | int | 0% | 0–5 | Goals scored by this player in this match. Mean 0.07 — most rows are 0. |
| 20 | `assists` | int | 0% | 0–4 | Goal assists (final pass before a goal) credited to this player. |
| 21 | `passes_total` | int | 0% | 0–188 | Total passes attempted. Highly position-dependent (defenders/DMs far more than forwards). |
| 22 | `passes_key` | int | 0% | 0–10 | Key passes: passes that directly created a goal-scoring opportunity. |
| 23 | `passes_accuracy` | float | 0% | 0–181 | **⚠ Data quality issue**: values above 100 indicate some rows contain a total count rather than a percentage. Clip to 0–100 range before use. |
| 24 | `dribbles_attempts` | int | 0% | 0–20 | Number of times the player attempted to dribble past an opponent. |
| 25 | `dribbles_success` | int | 0% | 0–15 | Successful dribbles (opponent beaten). Success rate = `dribbles_success / dribbles_attempts`. |
| 26 | `tackles_total` | int | 0% | 0–12 | Total tackle attempts by this player. |
| 27 | `tackles_blocks` | int | 0% | 0–7 | Shots blocked by this player. Often grouped with tackles in API data. |
| 28 | `tackles_interceptions` | int | 0% | 0–13 | Passes intercepted by this player. |
| 29 | `duels_total` | int | 0% | 0–45 | Total physical/aerial duels contested. |
| 30 | `duels_won` | int | 0% | 0–22 | Duels won. Win rate = `duels_won / duels_total`. |
| 31 | `fouls_drawn` | int | 0% | 0–8 | Fouls won by this player. |
| 32 | `fouls_committed` | int | 0% | 0–8 | Fouls committed by this player. |
| 33 | `cards_yellow` | int | 0% | 0–2 | Yellow cards received. 2 = sent off via two yellows. |
| 34 | `cards_red` | int | 0% | 0–1 | Straight red cards. Very rare (~0.2% of appearances). |

---

## GROUP 4 — Match Context (cols 35–49)
*What was the match situation for this player's team?*
**Source**: API team results (joined by `fixture_id`). **Coverage**: ~100% for cols 35–46; 100% null for cols 47–49.

| # | Column | Type | Null% | Range | Meaning |
|---|--------|------|-------|-------|---------|
| 35 | `is_home` | bool | 0% | 0/1 | 1 if `player_team` was the home side. Exactly 50% of rows are home. |
| 36 | `opponent_team` | str | 0% | — | The team the player faced. |
| 37 | `goals_for` | int | 0% | 0–9 | Goals scored by `player_team`. |
| 38 | `goals_against` | int | 0% | 0–9 | Goals conceded by `player_team`. |
| 39 | `result` | str | 0% | Win/Loss/Draw | Match outcome from `player_team`'s perspective. |
| 40 | `points` | int | 0% | 0/1/3 | League points earned (3 = win, 1 = draw, 0 = loss). |
| 41 | `team_shots_on_goal` | float | 0% | 0–17 | Shots on target by `player_team`. |
| 42 | `team_total_shots` | float | 0% | 0–38 | Total shot attempts by `player_team`. |
| 43 | `team_possession` | float | 0% | 0–100 | Ball possession % for `player_team`. |
| 44 | `team_corner_kicks` | float | 0.1% | 0–20 | Corner kicks won by `player_team`. |
| 45 | `team_fouls` | float | 0% | 1–25 | Total fouls committed by `player_team`. |
| 46 | `team_gk_saves` | float | 0.3% | 0–13 | Saves made by `player_team`'s goalkeeper. |
| 47 | `opp_shots_on_goal` | float | **100%** | — | **⛔ EMPTY — drop before training.** Never populated. |
| 48 | `opp_total_shots` | float | **100%** | — | **⛔ EMPTY — drop before training.** Never populated. |
| 49 | `opp_possession` | float | **100%** | — | **⛔ EMPTY — drop before training.** Compute as `100 − team_possession` if needed. |

---

## GROUP 5 — Workload & Fatigue Features (cols 50–56)
*How fresh or fatigued is this player heading into this match?*
**Source**: Recomputed in Section 9c from `date + minutes_played + is_home` for all 68,643 rows. Originally SofaScore PL-only (74.6% null); now full coverage.

| # | Column | Type | Null% | Range | Meaning |
|---|--------|------|-------|-------|---------|
| 50 | `ss_minutes` | float | 0% | 0–120 | Minutes played per SofaScore, backfilled with `minutes_played` where SofaScore was absent. Effectively identical to `minutes_played` for non-PL rows. |
| 51 | `sofascore_rating` | float | **81.7%** | 3–10 | SofaScore's independent per-match player rating. Available only for **Premier League** fixtures. Not used as the primary target — see `next_api_rating` (col 77) and `api_rating_decline_flag` (col 78). |
| 52 | `rest_days` | float | **10%** | 0–21 | Days between the player's previous fixture and this match. Clipped at 21. **NaN for a player's first appearance** (10% = ~6,895 first appearances). Mean = 8.9 days; median = 7 days. |
| 53 | `high_congestion_flag` | float | 0% | 0/1 | 1 if `rest_days ≤ 6` — two matches within a 7-day window. **41% of all appearances** are flagged. First appearances (NaN rest_days) assigned 0. |
| 54 | `min_last_7d` | float | 0% | 0–215 | Total minutes played across all competitions in the 7 days before this match. Captures acute workload. Mean = 34.8 min. |
| 55 | `acwr_ratio` | float | 0% | 0–4.0 | **Acute:Chronic Workload Ratio** = `min_last_7d ÷ (min_last_28d ÷ 4)`. Values > 1.5 are flagged as elevated injury risk in sports science literature. Clipped at 4.0. Mean = 0.75. |
| 56 | `consecutive_away_games` | float | 0% | 0–7 | Consecutive away matches played by this player up to and including this appearance. Resets to 0 on any home game. Max observed: 7. Mean = 0.68. |

---

## GROUP 6 — FBRef Detailed Action Stats (cols 57–67)
*Granular per-action statistics from FBRef match reports.*
**Source**: FBRef scraped data. **Coverage**: ~10% (4 CL teams per season only).

| # | Column | Type | Null% | Range | Meaning |
|---|--------|------|-------|-------|---------|
| 57 | `fb_min` | float | 89.8% | 1–120 | Minutes played per FBRef. Use as a consistency check against `minutes_played`. |
| 58 | `fb_goals` | float | 89.8% | 0–5 | Goals per FBRef. Should match `goals` for rows where both are present. |
| 59 | `fb_assists` | float | 89.8% | 0–4 | Assists per FBRef. |
| 60 | `fb_shots` | float | 89.9% | 0–10 | Total shots per FBRef. |
| 61 | `fb_sot` | float | 89.9% | 0–8 | Shots on target per FBRef. |
| 62 | `fb_tackles_won` | float | 89.9% | 0–8 | **Successful tackles only** — more precise than API `tackles_total`. |
| 63 | `fb_crosses` | float | 89.9% | 0–22 | Cross attempts into the box. |
| 64 | `fb_interceptions` | float | 89.9% | 0–7 | Passes intercepted per FBRef. |
| 65 | `fb_fouls` | float | 89.9% | 0–7 | Fouls committed per FBRef. |
| 66 | `fb_fouled` | float | 89.9% | 0–7 | Times fouled per FBRef. |
| 67 | `fb_offsides` | float | 89.9% | 0–5 | Offside traps caught. |

> **Training note**: 89.9% null. For the general model, either impute with API equivalents or train on the 10% CL-team subset as a richer feature set.

---

## GROUP 7 — Squad Injury Context (cols 68–70)
*How many of this player's teammates were injured for this specific match?*
**Source**: `team_match_injury_outcomes` (joined by `date + player_team`). **Coverage**: 66.4% non-null (PL matches only).

| # | Column | Type | Null% | Range | Meaning |
|---|--------|------|-------|-------|---------|
| 68 | `squad_injured_count` | float | 33.6% | 0–12 | Squad members missing this match due to physical injury or illness. Mean = 3.5. High values force fatigued players to play more minutes. |
| 69 | `squad_soft_tissue_count` | float | 33.6% | 0–7 | Subset of `squad_injured_count` — soft-tissue injuries (hamstring, calf, quad, adductor). Load-related; often signals overtraining across the squad. |
| 70 | `squad_avg_days_out` | float | 33.6% | 0–307 | Average reported recovery time (days) for the injured squad members on this matchday. Skewed by long-term injuries (mean ~150 days). |

---

## GROUP 8 — Legacy SofaScore Target Variables (cols 71–72)
*Original targets computed from SofaScore ratings. Kept for PL-only submodel validation and cross-checking. **Not recommended as primary targets** due to severe coverage limitations.*

| # | Column | Type | Null% | Range | Meaning |
|---|--------|------|-------|-------|---------|
| 71 | `next_sofascore_rating` | float | **82.3%** | 3–10 | The SofaScore rating the same player received in their next match. Available for PL rows only. Use `next_api_rating` (col 77) as the primary regression target. |
| 72 | `rating_decline_flag` | int | 0% | 0/1 | 1 if `sofascore_rating` drops > 0.5 in the next match. Only **2.3% positive** (1,603 rows); non-PL rows forced to 0. `scale_pos_weight ≈ 41.8`. Replaced by `api_rating_decline_flag` (col 78) as the primary classification target. |

---

## GROUP 9 — Lag Injury Features (cols 73–76)
*Was this player recently returning from injury? How long since they last missed matches?*
**Source**: Computed in Section 9b from `ALL_TEAMS_injuries_days_out` using binary-search lookback over physical absence records.

| # | Column | Type | Null% | Range | Meaning |
|---|--------|------|-------|-------|---------|
| 73 | `fixtures_missed_last_30d` | int | 0% | 0–7 | Fixtures missed due to physical injury or illness in the 30 days before this match. Computed via lag lookback (absence records are mutually exclusive with the master table which contains only played games). 0.3% of rows > 0. |
| 74 | `fixtures_missed_last_90d` | int | 0% | 0–14 | Same over a 90-day window. Captures longer injury histories. 0.9% of rows > 0. |
| 75 | `returning_from_injury` | int | 0% | 0/1 | 1 if `fixtures_missed_last_30d > 0` — player missed ≥1 fixture due to injury in the prior 30 days. Only **0.3%** positive. High-signal feature when it fires. |
| 76 | `days_since_last_injury` | float | **98.3%** | 3–761 | Days since the most recent injury absence before this match. 98.3% null — no injury history available for most players. Mean ≈ 140 days where non-null. **Drop before training.** |

---

## GROUP 10 — Primary API Target Variables (cols 77–78)
*What the model is trained to predict. Built from the API `rating` column (all competitions, all teams).*
**Source**: Computed in Section 9d (API rating) + Section 9e (minutes guard). Group key: normalised `(player_name + player_team)` — avoids the `player_id = 0` collision.

| # | Column | Type | Null% | Range | Meaning |
|---|--------|------|-------|-------|---------|
| 77 | `next_api_rating` | float | **32.7%** | 3–10 | API rating the same player received in their **next match** (rating = 0 treated as null). **Primary regression target.** 67.3% fill — far better than `next_sofascore_rating` (17.7%). Null for last appearances and for rows where neither current nor next match has a real rating. |
| 78 | `api_rating_decline_flag` | int | 0% | 0/1 | **Primary classification target.** 1 if the player's API rating drops > 0.5 points in the next match, **subject to both matches having ≥ 45 minutes played**. The 45-minute guard prevents role-change artifacts (Starter→Sub transitions inflated the rate by 6.5 pp when unguarded). **9.8% positive** (6,729 rows). `scale_pos_weight ≈ 9.2`. |

> **Threshold sensitivity** (`api_rating_decline_flag`): 0.30 → 18.9% | **0.50 → 9.8%** (default) | 0.75 → 8.2% | 1.00 → 4.3%. Change `DECLINE_THRESHOLD` in Section 9d and re-run 9d + 9e.

---

## GROUP 11 — Next Match Context Features (cols 79–80)
*What role did this player have in their next match?*
**Source**: Computed in Section 9e via `groupby + shift(-1)` on `is_substitute` and `minutes_played`. **Coverage**: 90% non-null (10% null = last appearance per player-team pair).

| # | Column | Type | Null% | Range | Meaning |
|---|--------|------|-------|-------|---------|
| 79 | `next_is_substitute` | bool | **10%** | 0/1 | 1 if the player came off the bench in their next match. Null for last appearances. Allows the model to learn role-change patterns independently of the target variable. A starter→sub transition is itself a form of reduced opportunity, separate from pure performance quality. |
| 80 | `next_minutes_played` | float | **10%** | 0–120 | Minutes played in the next match. Null for last appearances. Used internally as the minutes guard for `api_rating_decline_flag`; also a direct feature for predicting workload continuity. |

---

## Summary: Coverage & Training Guidance

| Group | Cols | Coverage | Training Recommendation |
|-------|------|----------|------------------------|
| Match Identity | 1–8 | 100% | Use `date`, `competition`, `season`; drop `fixture_id`, `round` |
| Player Identity | 9–12 | 100% | Encode `player_position`; drop `player_id`, `player_number` |
| Individual Performance | 13–34 | 100% | ✅ All usable. Treat `rating = 0` as null (col 14). |
| Match Context | 35–46 | ~100% | ✅ Direct features; encode `result` |
| opp_* columns | 47–49 | **0%** | ⛔ **Drop** — entirely empty |
| Workload & Fatigue | 50–56 | 0–82% | ✅ cols 52–56: 0–10% null; col 51 PL-only (not a target) |
| FBRef Stats | 57–67 | ~10% | ⚠ Impute with API equivalents or use CL-team subset only |
| Squad Injury Context | 68–70 | 66% | ✅ Impute with median (~3.5 injuries) |
| Legacy SofaScore Targets | 71–72 | 18–100% | ℹ️ PL validation only; use col 78 for training |
| Lag Injury Features | 73–75 | 100% | ✅ All usable (low signal but zero cost) |
| Lag Injury (days) | 76 | **1.7%** | ⛔ **Drop** — 98.3% null |
| **Primary API Targets** | **77–78** | **67–100%** | 🎯 **col 78 = classifier target; col 77 = regression target** |
| Next Match Context | 79–80 | 90% | ✅ Features for role-change modelling; not target leakage (future data but conceptually useful for shift analysis) |